In [1]:
# Import required libraries
import tensorflow as tf  # Deep learning framework
from tensorflow.keras.models import Sequential  # To build sequential CNN model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout  # CNN layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # For image preprocessing
import matplotlib.pyplot as plt  # For plotting graphs

train_path = "Plant_Disease_Dataset/train"
valid_path = "Plant_Disease_Dataset/valid"
test_path  = "Plant_Disease_Dataset/test"
# -------------------- DATA PREPROCESSING --------------------

# Create training data generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,          # Normalize pixel values (0-255 → 0-1)
    shear_range=0.2,         # Apply shear transformation (tilt images)
    zoom_range=0.2,          # Zoom images randomly
    horizontal_flip=True     # Flip images horizontally
)

# Create testing data generator (only normalization)
test_datagen = ImageDataGenerator(rescale=1./255)

# Load training dataset from directory
training_set = train_datagen.flow_from_directory(
    'dataset/train',         # Path to training dataset folder
    target_size=(128, 128),  # Resize images to 128x128
    batch_size=32,           # Number of images per batch
    class_mode='categorical' # Multi-class classification
)

# Load testing dataset from directory
test_set = test_datagen.flow_from_directory(
    'dataset/test',          # Path to testing dataset folder
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

# -------------------- BUILDING CNN MODEL --------------------

# Initialize CNN model
cnn = Sequential()

# First Convolution Layer
cnn.add(Conv2D(
    filters=32,              # Number of filters (feature detectors)
    kernel_size=3,           # Size of filter (3x3)
    activation='relu',       # Activation function
    input_shape=[128, 128, 3] # Input image shape (RGB image)
))

# First Pooling Layer
cnn.add(MaxPooling2D(
    pool_size=2,             # Pooling size (2x2)
    strides=2                # Step size
))

# Second Convolution Layer
cnn.add(Conv2D(
    filters=32,
    kernel_size=3,
    activation='relu'
))

# Second Pooling Layer
cnn.add(MaxPooling2D(
    pool_size=2,
    strides=2
))

# Flatten layer (convert 2D feature maps into 1D vector)
cnn.add(Flatten())

# Fully connected layer
cnn.add(Dense(
    units=128,               # Number of neurons
    activation='relu'
))

# Dropout layer to prevent overfitting
cnn.add(Dropout(0.5))       # Randomly disables 50% neurons during training

# Output layer
cnn.add(Dense(
    units=training_set.num_classes, # Number of output classes
    activation='softmax'            # For multi-class classification
))

# -------------------- COMPILING MODEL --------------------

cnn.compile(
    optimizer='adam',              # Optimizer to update weights
    loss='categorical_crossentropy', # Loss function for multi-class
    metrics=['accuracy']           # Evaluation metric
)

# -------------------- TRAINING MODEL --------------------

history = cnn.fit(
    x=training_set,           # Training data
    validation_data=test_set, # Validation data
    epochs=10                 # Number of iterations
)

# -------------------- VISUALIZATION --------------------

# Plot training and validation accuracy
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='validation accuracy')
plt.legend()
plt.title('Accuracy')
plt.show()

# Plot training and validation loss
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='validation loss')
plt.legend()
plt.title('Loss')
plt.show()

# -------------------- SAVE MODEL --------------------

cnn.save('plant_disease_model.h5')  # Save trained model to file

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset/train'